In [ ]:
!apt install fluidsynth

!git clone https://github.com/jthickstun/anticipation.git
!pip install ./anticipation
!pip install -r anticipation/requirements.txt

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fluid-soundfont-gm libevdev2 libfluidsynth3 libgudev-1.0-0 libinput-bin
  libinput10 libinstpatch-1.0-2 libmd4c0 libmtdev1 libqt5core5a libqt5dbus5
  libqt5gui5 libqt5network5 libqt5svg5 libqt5widgets5 libwacom-bin
  libwacom-common libwacom9 libxcb-icccm4 libxcb-image0 libxcb-keysyms1
  libxcb-render-util0 libxcb-util1 libxcb-xinerama0 libxcb-xinput0 libxcb-xkb1
  libxkbcommon-x11-0 qsynth qt5-gtk-platformtheme qttranslations5-l10n
  timgm6mb-soundfont
Suggested packages:
  fluid-soundfont-gs qt5-image-formats-plugins qtwayland5 jackd
The following NEW packages will be installed:
  fluid-soundfont-gm fluidsynth libevdev2 libfluidsynth3 libgudev-1.0-0
  libinput-bin libinput10 libinstpatch-1.0-2 libmd4c0 libmtdev1 libqt5core5a
  libqt5dbus5 libqt5gui5 libqt5network5 libqt5svg5 libqt5widgets5 libwacom-bin
  libwacom-common libwacom9 libx

In [ ]:
import sys,time
import numpy as np

import midi2audio
import transformers
import os

import torch
import torch.nn.functional as F

import random

from transformers import AutoModelForCausalLM
from transformers import BertConfig, BertModel
from pathlib import Path
from IPython.display import Audio

from anticipation import ops
from anticipation.sample import generate
from anticipation.tokenize import extract_instruments
from anticipation.convert import events_to_midi,midi_to_events
from anticipation.config import *
from anticipation.vocab import *


os.environ['TORCH_USE_CUDA_DSA'] = '1'
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
''' load our model '''

import torch
from transformers import BertConfig, BertModel

# set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# rebuild configuration
configuration = BertConfig()
configuration.vocab_size = 55128
configuration.max_position_embeddings = 2048

# load model weights
structure_derivation_model = BertModel(configuration).to(device)

# load the saved state
checkpoint_path = "/content/drive/MyDrive/MusicData/bert_checkpoints/structure_derivation_model.pth"
structure_derivation_model.load_state_dict(torch.load(checkpoint_path, map_location=device))

# set to evaluation mode
structure_derivation_model.eval()


Using device: cuda


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(55128, 768, padding_idx=0)
    (position_embeddings): Embedding(2048, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
 

In [ ]:
''' load each AMT model configuration '''

SMALL_MODEL = 'stanford-crfm/music-small-800k'     # faster inference, worse sample quality
MEDIUM_MODEL = 'stanford-crfm/music-medium-800k'   # slower inference, better sample quality
LARGE_MODEL = 'stanford-crfm/music-large-800k'     # slowest inference, best sample quality

# Load AMT models
small_model = AutoModelForCausalLM.from_pretrained(SMALL_MODEL).cuda()
medium_model = AutoModelForCausalLM.from_pretrained(MEDIUM_MODEL).cuda()
large_model = AutoModelForCausalLM.from_pretrained(LARGE_MODEL).cuda()

# a MIDI synthesizer
fs = midi2audio.FluidSynth('/usr/share/sounds/sf2/FluidR3_GM.sf2')

# the MIDI synthesis script
def synthesize(fs, tokens):
    mid = events_to_midi(tokens)
    mid.save('tmp.mid')
    fs.midi_to_audio('tmp.mid', 'tmp.wav')
    return 'tmp.wav'

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/512M [00:00<?, ?B/s]

Some weights of the model checkpoint at stanford-crfm/music-small-800k were not used when initializing GPT2LMHeadModel: ['token_out_embeddings']
- This IS expected if you are initializing GPT2LMHeadModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing GPT2LMHeadModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.44G [00:00<?, ?B/s]

Some weights of the model checkpoint at stanford-crfm/music-medium-800k were not used when initializing GPT2LMHeadModel: ['token_out_embeddings']
- This IS expected if you are initializing GPT2LMHeadModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing GPT2LMHeadModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.12G [00:00<?, ?B/s]

In [ ]:
''' load lakh midi dataset '''

import kagglehub
from sklearn.model_selection import train_test_split
from pathlib import Path

# download latest version of clean lakh dataset
path = kagglehub.dataset_download("imsparsh/lakh-midi-clean")
midi_paths=Path(path)
print("Path to dataset files:", midi_paths)

# list all .mid files
midi_files = list(midi_paths.rglob("*.mid"))

# read all the cleaned filepaths
path = '/content/drive/MyDrive/MusicData/clean_midi_files.txt'
lm_midi_files = []
with open(path, 'r') as f:
    for line in f:
        lm_midi_files.append(Path(line.strip()))

print (lm_midi_files[1])

100%|██████████| 226M/226M [00:01<00:00, 235MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/imsparsh/lakh-midi-clean/versions/1


In [ ]:
''' load symphonynet dataset '''

from pathlib import Path

# access symphonynet data
sn_path = Path('/content/drive/MyDrive/MusicData/SymphonyNet_Dataset/contemporary')
sn_midi_files = list(sn_path.rglob("*.mid"))

print(sn_midi_files[1])


/content/drive/MyDrive/MusicData/SymphonyNet_Dataset/contemporary/29789.mid


In [ ]:
''' set config. for evaluation ''''

vocab_size = structure_derivation_model.config.vocab_size
print (vocab_size)

In [ ]:
''' create necessary assistant functions '''

# function to generate prompts
def extract_prompts(lm_midi_files, sn_midi_files, prompt_length, num_prompts,start_time=0):
  prompts = []
  random_lm = random.sample(lm_midi_files, num_prompts//2)
  random_sn = random.sample(sn_midi_files, num_prompts//2)
  random_midi_files = random_lm + random_sn

  for midi_file in random_midi_files:
    events = midi_to_events(str(midi_file))
    prompt = ops.clip(events, 0, prompt_length)
    prompts.append(prompt)

  return prompts

# function to generate music using prompts
def generate_music(model, prompt, prompt_length, generated_music_length):
  history = prompt.copy()
  proposal = generate(model, start_time=prompt_length, end_time=prompt_length+generated_music_length, inputs=history, top_p=0.98, debug=False)
  mid = events_to_midi(proposal)
  return mid

# function to segment a generated piece into 10-second chunks
def segment(sequence, total_duration, segment_length):
  tokens_per_second = len(sequence) / total_duration
  tokens_per_segment = int(tokens_per_second * segment_length)

  segments = []
  for i in range(0, len(sequence), tokens_per_segment):
      segment = sequence[i:i + tokens_per_segment]
      if len(segment) == tokens_per_segment:
          segments.append(segment)

  return segments

In [ ]:
''' create eval loop '''

from tqdm import tqdm
import json

def _evaluate(model,model_name, prompts, structure_derivation_model, checkpoint_path="/content/drive/MyDrive/MusicData/results.json"):
  model.eval()
  structure_derivation_model.eval()

  all_scores = []
  prompt_length = 10
  generation_length = 60
  MAX_TOKENS = 2048

  # load existing results if a checkpoint exists
  try:
    with open(checkpoint_path, 'r') as f:
      results = json.load(f)
  except FileNotFoundError:
    results = {}

  if model_name in results:
    print(f"Skipping {model_name} as it's already evaluated.")
    return results[model_name]

  for prompt in tqdm(prompts, desc=f" evaluating {model_name}"):
    # generate song using prompt
    mid = generate_music(model, prompt, prompt_length, generation_length)

    # tokenise the generated music piece
    tokenised_midi = midi_to_events(mid)

    # split generation into segments
    segments = segment(tokenised_midi, generation_length, prompt_length)

    # filter out samples that are None or empty
    if not segments:
        continue

    # skip if too few segments
    if len(segments) < 2:
      continue

    # truncate each segment if necessary
    truncated_segments = [s[:MAX_TOKENS] for s in segments]

    # convert each segment to a tensor
    input_ids_list = [torch.tensor(s).unsqueeze(0).to(device) for s in truncated_segments]

    # final check, extract any invalid tokens
    final_input_ids_list = [s for s in input_ids_list if not ((s >= vocab_size).any() or (s < 0).any())]


    # pass each segment through the SD model
    with torch.no_grad():
      embeddings = []
      for input_ids in final_input_ids_list:
          output = structure_derivation_model(input_ids=input_ids)
          emb = output.last_hidden_state[:, 0]  # CLS token
          emb = F.normalize(emb, dim=-1)        # normalise output
          embeddings.append(emb)

    # define anchor ad comparison segments
    anchor = embeddings[0]
    others = embeddings[1:]

    try: # compute cosine similarity
      scores = [
          F.cosine_similarity(anchor, other, dim=-1).item()
          for other in others
      ]

      print (scores)

      # add the mean score to 'all scores' array
      all_scores.append(np.mean(scores))

    except:
      print ("invalid tokens")
      continue


  mean_score = np.mean(all_scores)

  # Save the result for the current model
  results[model_name] = mean_score
  with open(checkpoint_path, 'w') as f:
    json.dump(results, f)

  return mean_score

In [ ]:
# generate 30 different 10 second prompts
prompts = extract_prompts(lm_midi_files, sn_midi_files, prompt_length=10, num_prompts=30)


Prompts saved to drive


In [ ]:
''' run eval loop '''

results = {}

models_and_prompts = [
    ("Small Model", small_model, prompts),
    ("Medium Model", medium_model, prompts),
    ("Large Model", large_model, prompts)
]

for name, amt_model, prompts in models_and_prompts:
    score = _evaluate(amt_model,name, prompts, structure_derivation_model,"/content/drive/MyDrive/MusicData/final_results.json")
    results[name] = score
    print(f"{name}: mean similarity = {score:.4f}")

Skipping Small Model as it's already evaluated.
Small Model: mean similarity = 0.7255
Skipping Medium Model as it's already evaluated.
Medium Model: mean similarity = 0.7316


 evaluating Large Model:   3%|▎         | 1/30 [23:05<11:09:49, 1385.83s/it]

[0.7843321561813354, 0.7185784578323364, 0.7774900197982788, 0.7651299238204956, 0.7821270823478699]



100%|█████████▉| 5985/6000 [09:18<00:02,  7.48it/s]
6002it [09:23, 10.66it/s]
 evaluating Large Model:   7%|▋         | 2/30 [32:29<7:00:57, 902.06s/it]  

[0.8341271877288818, 0.7159064412117004, 0.7508955597877502, 0.8172503113746643, 0.810559868812561]


...

[0.8799649477005005, 0.9017406702041626, 0.8835464715957642, 0.7721507549285889, 0.7886772751808167]



 evaluating Large Model:  97%|█████████▋| 29/30 [5:11:54<08:19, 499.47s/it]

[0.8876602649688721, 0.8448883295059204, 0.8420027494430542, 0.8534191846847534, 0.9079192280769348]



100%|██████████| 6000/6000 [06:03<00:00, 16.63it/s]
6023it [06:07, 11.40it/s]                          
6045it [06:14, 16.14it/s]
 evaluating Large Model: 100%|██████████| 30/30 [5:18:08<00:00, 636.29s/it]

[0.8995382785797119, 0.8717123866081238, 0.8911179900169373, 0.8394426107406616, 0.8532320857048035]
Large Model: mean similarity = 0.7800
